# KosWatt - Smart Energy Agentic AI
**Notebook: Integrated AI Logic Engine**

This notebook defines the core intelligence of KosWatt:
- Fuzzy Logic system for waste level assessment
- STRIPS Planning engine for autonomous action sequencing
- Main agent function integrating both components

In [ ]:
# ─── Dependencies ─────────────────────────────────────────────────────────────
import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

## 1. Fuzzy Universe Definition

Two input variables drive the core waste assessment:
- `watt`: total power consumption (0-550 W)
- `occupancy`: room occupancy flag (0 = empty, 1 = occupied)

Two contextual variables shape the ethical guardrails:
- `temperature`: ambient temperature (20-40 C)
- `time_of_day`: 0 = day, 1 = night

One output variable:
- `waste_level`: waste score 0-100

In [ ]:
# ─── Fuzzy Universes ──────────────────────────────────────────────────────────
watt_range        = np.arange(0, 551, 1)
occupancy_range   = np.arange(0, 1.01, 0.01)
temperature_range = np.arange(20, 41, 1)
tod_range         = np.arange(0, 1.01, 0.01)   # time_of_day
waste_range       = np.arange(0, 101, 1)

# ─── Membership Functions: watt ───────────────────────────────────────────────
watt_low    = fuzz.trimf(watt_range, [0,   0,   200])
watt_medium = fuzz.trimf(watt_range, [100, 275, 400])
watt_high   = fuzz.trimf(watt_range, [300, 550, 550])

# ─── Membership Functions: occupancy ─────────────────────────────────────────
occ_empty    = fuzz.trimf(occupancy_range, [0, 0, 0.5])
occ_occupied = fuzz.trimf(occupancy_range, [0.5, 1, 1])

# ─── Membership Functions: temperature ───────────────────────────────────────
temp_cool = fuzz.trimf(temperature_range, [20, 20, 30])
temp_hot  = fuzz.trimf(temperature_range, [28, 40, 40])

# ─── Membership Functions: time_of_day ───────────────────────────────────────
tod_day   = fuzz.trimf(tod_range, [0, 0, 0.5])
tod_night = fuzz.trimf(tod_range, [0.5, 1, 1])

# ─── Membership Functions: waste_level (output) ───────────────────────────────
waste_low    = fuzz.trimf(waste_range, [0,   0,   40])
waste_medium = fuzz.trimf(waste_range, [25,  50,  75])
waste_high   = fuzz.trimf(waste_range, [60, 100, 100])

print("Fuzzy universes and membership functions defined.")

## 2. Fuzzy Rule Engine

Rules are categorized into two device groups:

**Rigid/Non-Critical (TV):** No tolerance when room is empty. Any active TV in an unoccupied room = HIGH waste.

**Adaptive/Critical (AC & Lamp):** Ethical guardrails apply:
- AC tolerates hot ambient conditions to prevent thermal discomfort on return.
- Lamp at night in an empty room receives a safety tolerance buffer.

In [ ]:
def compute_waste_score(watt_val, occupancy_val, temperature_val, time_of_day_val):
    """
    Core fuzzy inference engine.
    
    All rule activations are computed manually (Mamdani-style) using skfuzzy
    membership interpolation, then aggregated via maximum, and defuzzified
    with the centroid method.

    Parameters
    ----------
    watt_val        : float  - current total power draw in Watts
    occupancy_val   : float  - 0 (empty) to 1 (occupied)
    temperature_val : float  - ambient temperature in Celsius
    time_of_day_val : float  - 0 (day) to 1 (night)

    Returns
    -------
    float - defuzzified waste score between 0 and 100
    """

    # Fuzzify all inputs
    mu_watt_low    = fuzz.interp_membership(watt_range, watt_low,    watt_val)
    mu_watt_medium = fuzz.interp_membership(watt_range, watt_medium, watt_val)
    mu_watt_high   = fuzz.interp_membership(watt_range, watt_high,   watt_val)

    mu_occ_empty    = fuzz.interp_membership(occupancy_range, occ_empty,    occupancy_val)
    mu_occ_occupied = fuzz.interp_membership(occupancy_range, occ_occupied, occupancy_val)

    mu_temp_cool = fuzz.interp_membership(temperature_range, temp_cool, temperature_val)
    mu_temp_hot  = fuzz.interp_membership(temperature_range, temp_hot,  temperature_val)

    mu_tod_day   = fuzz.interp_membership(tod_range, tod_day,   time_of_day_val)
    mu_tod_night = fuzz.interp_membership(tod_range, tod_night, time_of_day_val)

    # ─── Rule Set ─────────────────────────────────────────────────────────────
    # R1: Room occupied + high watt => waste HIGH (occupied overuse)
    r1 = np.fmin(mu_occ_occupied, mu_watt_high)

    # R2: Room occupied + low/medium watt => waste LOW (normal usage)
    r2 = np.fmin(mu_occ_occupied, np.fmax(mu_watt_low, mu_watt_medium))

    # R3 [TV - Rigid]: Room empty + high watt => waste HIGH (no tolerance)
    r3 = np.fmin(mu_occ_empty, mu_watt_high)

    # R4 [TV - Rigid]: Room empty + medium watt => waste HIGH
    r4 = np.fmin(mu_occ_empty, mu_watt_medium)

    # R5 [AC - Ethical]: Room empty + hot temperature => waste MEDIUM
    # Rationale: AC keeps room thermally tolerable for returning occupant
    r5 = np.fmin(mu_occ_empty, mu_temp_hot)

    # R6 [AC - Ethical]: Room empty + cool temperature => waste HIGH
    # Rationale: No thermal justification for running AC
    r6 = np.fmin(mu_occ_empty, mu_temp_cool)

    # R7 [Lamp - Ethical]: Room empty + daytime => waste HIGH
    # Rationale: Natural light available, no safety reason
    r7 = np.fmin(mu_occ_empty, mu_tod_day)

    # R8 [Lamp - Ethical]: Room empty + nighttime => waste MEDIUM
    # Rationale: Safety/deterrence lighting has partial justification
    r8 = np.fmin(mu_occ_empty, mu_tod_night)

    # R9: Room empty + low watt => waste LOW (standby only)
    r9 = np.fmin(mu_occ_empty, mu_watt_low)

    # ─── Aggregate consequents ────────────────────────────────────────────────
    # Map rules to output membership functions via minimum (Mamdani)
    agg_low    = np.fmax(
        np.fmin(r2, waste_low),
        np.fmin(r9, waste_low)
    )
    agg_medium = np.fmax(
        np.fmin(r5, waste_medium),
        np.fmin(r8, waste_medium)
    )
    agg_high   = np.fmax(
        np.fmax(
            np.fmin(r1, waste_high),
            np.fmin(r3, waste_high)
        ),
        np.fmax(
            np.fmin(r4, waste_high),
            np.fmax(
                np.fmin(r6, waste_high),
                np.fmin(r7, waste_high)
            )
        )
    )

    # Final aggregation across all output levels
    aggregated = np.fmax(agg_low, np.fmax(agg_medium, agg_high))

    # ─── Defuzzification (Centroid) ───────────────────────────────────────────
    if aggregated.max() == 0:
        return 0.0

    score = fuzz.defuzz(waste_range, aggregated, 'centroid')
    return float(np.clip(score, 0, 100))


print("Fuzzy inference engine ready.")

## 3. STRIPS Planning Engine

STRIPS (Stanford Research Institute Problem Solver) models the problem as:
- **State**: dictionary of device statuses
- **Goal State**: energy-efficient configuration
- **Operators**: discrete actions with preconditions and effects

The planner uses forward-chaining BFS to find the shortest action sequence from current state to goal state.

In [ ]:
from collections import deque


class StripsPlanner:
    """
    STRIPS-based autonomous action planner for KosWatt.

    Device states use boolean flags:
      ac_on    : bool - AC unit status
      lamp_on  : bool - Lamp status
      tv_on    : bool - TV status
      ac_eco   : bool - AC in ECO mode (lower consumption)

    The planner selects the appropriate goal state based on
    fuzzy context (temperature, time_of_day) to respect
    ethical guardrails embedded in the fuzzy rule set.
    """

    def __init__(self):
        # Each action: name, precondition (fn), effect (fn)
        self.operators = [
            {
                'name': 'TURN_OFF_TV',
                'precondition': lambda s: s.get('tv_on', False),
                'effect': lambda s: {**s, 'tv_on': False}
            },
            {
                'name': 'TURN_OFF_LAMP',
                'precondition': lambda s: s.get('lamp_on', False),
                'effect': lambda s: {**s, 'lamp_on': False}
            },
            {
                'name': 'TURN_OFF_AC',
                'precondition': lambda s: s.get('ac_on', False) and not s.get('ac_eco', False),
                'effect': lambda s: {**s, 'ac_on': False, 'ac_eco': False}
            },
            {
                'name': 'SET_AC_TO_ECO',
                'precondition': lambda s: s.get('ac_on', False) and not s.get('ac_eco', False),
                'effect': lambda s: {**s, 'ac_eco': True}
            },
        ]

    def _build_goal_state(self, current_state, temperature, time_of_day):
        """
        Derive goal state from ethical guardrails:
        - TV always off in empty room (rigid device)
        - AC: ECO mode if hot, OFF if cool (adaptive device)
        - Lamp: allow ON at night for safety, OFF during day
        """
        goal = dict(current_state)

        # TV: rigid - always off
        goal['tv_on'] = False

        # AC: ethical guardrail based on temperature
        if goal.get('ac_on', False):
            if temperature >= 32:
                # Hot: keep running in ECO mode
                goal['ac_eco'] = True
            else:
                # Cool enough: turn off
                goal['ac_on'] = False
                goal['ac_eco'] = False

        # Lamp: ethical guardrail based on time of day
        if goal.get('lamp_on', False):
            if time_of_day == 0:   # daytime: no justification
                goal['lamp_on'] = False
            # nighttime: lamp stays on (safety tolerance)

        return goal

    def _state_satisfies_goal(self, state, goal):
        """Check if every key in goal matches the current state."""
        return all(state.get(k) == v for k, v in goal.items())

    def _state_to_key(self, state):
        """Convert state dict to hashable tuple for BFS visited set."""
        return tuple(sorted(state.items()))

    def plan(self, start_state, temperature, time_of_day):
        """
        BFS forward search from start_state to goal_state.

        Parameters
        ----------
        start_state  : dict  - current device statuses
        temperature  : float - ambient temperature
        time_of_day  : int   - 0 (day) or 1 (night)

        Returns
        -------
        list of str - ordered action names, empty if already at goal
        """
        goal_state = self._build_goal_state(start_state, temperature, time_of_day)

        if self._state_satisfies_goal(start_state, goal_state):
            return []

        queue   = deque([(start_state, [])])
        visited = {self._state_to_key(start_state)}

        while queue:
            current_state, actions = queue.popleft()

            for op in self.operators:
                if op['precondition'](current_state):
                    next_state = op['effect'](current_state)
                    next_key   = self._state_to_key(next_state)

                    if next_key not in visited:
                        new_actions = actions + [op['name']]

                        if self._state_satisfies_goal(next_state, goal_state):
                            return new_actions

                        visited.add(next_key)
                        queue.append((next_state, new_actions))

        # No plan found (should not occur with valid state space)
        return []


print("STRIPS Planner class defined.")

## 4. Main Agent Function

`core_koswatt_agent` is the single entry point used by the Streamlit dashboard.
It returns:
1. `fuzzy_score` - continuous waste score 0-100
2. `waste_category` - LOW / MEDIUM / HIGH WASTE
3. `action_sequence` - ordered list of STRIPS actions (empty if no intervention needed)

In [ ]:
# Power draw constants (Watts)
STANDBY_WATTS   = 10
LAMP_WATTS      = 20
TV_WATTS        = 80
AC_HOT_WATTS    = 400   # AC load when ambient > 30C
AC_COOL_WATTS   = 250   # AC load when ambient <= 30C
AC_ECO_WATTS    = 150   # AC load in ECO mode


def calculate_watt(status_ac, status_lamp, status_tv, temperature):
    """
    Dynamic watt calculation based on device switch states.
    AC load is adaptive: higher draw in hot conditions.
    """
    total = STANDBY_WATTS
    if status_lamp: total += LAMP_WATTS
    if status_tv:   total += TV_WATTS
    if status_ac:
        total += AC_HOT_WATTS if temperature > 30 else AC_COOL_WATTS
    return total


def calculate_post_action_watt(final_state, temperature):
    """
    Compute watt after STRIPS actions are applied.
    Accounts for ECO mode as a partial load reduction.
    """
    total = STANDBY_WATTS
    if final_state.get('lamp_on'): total += LAMP_WATTS
    if final_state.get('tv_on'):   total += TV_WATTS
    if final_state.get('ac_on'):
        if final_state.get('ac_eco'):
            total += AC_ECO_WATTS
        else:
            total += AC_HOT_WATTS if temperature > 30 else AC_COOL_WATTS
    return total


def apply_actions_to_state(start_state, actions):
    """
    Apply a sequence of STRIPS action names to a start state
    and return the resulting final device state.
    """
    planner    = StripsPlanner()
    op_map     = {op['name']: op for op in planner.operators}
    state      = dict(start_state)
    for action_name in actions:
        if action_name in op_map:
            state = op_map[action_name]['effect'](state)
    return state


def core_koswatt_agent(occupancy, temperature, time_of_day,
                       status_ac, status_lamp, status_tv):
    """
    Main KosWatt agent entry point.

    Parameters
    ----------
    occupancy    : int   - 0 (empty) or 1 (occupied)
    temperature  : float - room temperature in Celsius
    time_of_day  : int   - 0 (day) or 1 (night)
    status_ac    : bool  - AC currently on
    status_lamp  : bool  - Lamp currently on
    status_tv    : bool  - TV currently on

    Returns
    -------
    dict with keys:
      fuzzy_score      : float  - waste score 0-100
      waste_category   : str    - 'LOW WASTE' | 'MEDIUM WASTE' | 'HIGH WASTE'
      action_sequence  : list   - STRIPS action names ordered by priority
      initial_watt     : float  - total watt before AI intervention
      final_state      : dict   - device states after AI actions
      final_watt       : float  - estimated watt after AI intervention
    """
    #  Step 1: Dynamic watt calculation 
    initial_watt   = calculate_watt(status_ac, status_lamp, status_tv, temperature)
    occupancy_val  = float(occupancy)

    #  Step 2: Fuzzy inference 
    fuzzy_score    = compute_waste_score(
        watt_val        = initial_watt,
        occupancy_val   = occupancy_val,
        temperature_val = float(temperature),
        time_of_day_val = float(time_of_day)
    )

    #  Step 3: Classify waste category 
    if fuzzy_score > 65:
        waste_category = 'HIGH WASTE'
    elif fuzzy_score > 35:
        waste_category = 'MEDIUM WASTE'
    else:
        waste_category = 'LOW WASTE'

    #  Step 4: STRIPS Planning (only triggered on HIGH WASTE) 
    action_sequence = []
    start_state     = {
        'ac_on':   bool(status_ac),
        'lamp_on': bool(status_lamp),
        'tv_on':   bool(status_tv),
        'ac_eco':  False
    }

    if waste_category == 'HIGH WASTE':
        planner         = StripsPlanner()
        action_sequence = planner.plan(start_state, temperature, time_of_day)

    #  Step 5: Compute final state and post-action watt 
    final_state = apply_actions_to_state(start_state, action_sequence)
    final_watt  = calculate_post_action_watt(final_state, temperature)

    return {
        'fuzzy_score'    : round(fuzzy_score, 2),
        'waste_category' : waste_category,
        'action_sequence': action_sequence,
        'initial_watt'   : initial_watt,
        'final_state'    : final_state,
        'final_watt'     : final_watt
    }


print("core_koswatt_agent function ready.")

## 5. Validation Tests

Quick sanity checks covering the three ethical device categories.

In [ ]:
#  Test Cases 

test_cases = [
    {
        'label'       : 'Empty room, TV on, cool temp, daytime',
        'occupancy'   : 0, 'temperature': 24, 'time_of_day': 0,
        'status_ac'   : False, 'status_lamp': False, 'status_tv': True
    },
    {
        'label'       : 'Empty room, AC on, very hot temp',
        'occupancy'   : 0, 'temperature': 36, 'time_of_day': 0,
        'status_ac'   : True, 'status_lamp': False, 'status_tv': False
    },
    {
        'label'       : 'Empty room, AC on, cool temp',
        'occupancy'   : 0, 'temperature': 25, 'time_of_day': 1,
        'status_ac'   : True, 'status_lamp': False, 'status_tv': False
    },
    {
        'label'       : 'Empty room, lamp on, nighttime',
        'occupancy'   : 0, 'temperature': 28, 'time_of_day': 1,
        'status_ac'   : False, 'status_lamp': True, 'status_tv': False
    },
    {
        'label'       : 'Occupied room, all devices on',
        'occupancy'   : 1, 'temperature': 30, 'time_of_day': 1,
        'status_ac'   : True, 'status_lamp': True, 'status_tv': True
    },
]

print(f"{'Test Case':<50} {'Score':>6}  {'Category':<14}  Actions")
print("-" * 100)

for tc in test_cases:
    result = core_koswatt_agent(
        occupancy   = tc['occupancy'],
        temperature = tc['temperature'],
        time_of_day = tc['time_of_day'],
        status_ac   = tc['status_ac'],
        status_lamp = tc['status_lamp'],
        status_tv   = tc['status_tv']
    )
    actions = result['action_sequence'] if result['action_sequence'] else ['None']
    print(f"{tc['label']:<50} {result['fuzzy_score']:>6.1f}  {result['waste_category']:<14}  {', '.join(actions)}")

## 6. Membership Function Visualization

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle('KosWatt - Fuzzy Membership Functions', fontsize=14, fontweight='bold')

# Watt input
ax = axes[0, 0]
ax.plot(watt_range, watt_low,    label='Low',    color='#2ecc71')
ax.plot(watt_range, watt_medium, label='Medium', color='#f39c12')
ax.plot(watt_range, watt_high,   label='High',   color='#e74c3c')
ax.set_title('Input: Power Consumption (Watts)')
ax.set_xlabel('Watts'); ax.set_ylabel('Membership'); ax.legend(); ax.grid(alpha=0.3)

# Temperature
ax = axes[0, 1]
ax.plot(temperature_range, temp_cool, label='Cool', color='#3498db')
ax.plot(temperature_range, temp_hot,  label='Hot',  color='#e74c3c')
ax.set_title('Context: Temperature (Celsius)')
ax.set_xlabel('Degrees C'); ax.set_ylabel('Membership'); ax.legend(); ax.grid(alpha=0.3)

# Waste output
ax = axes[1, 0]
ax.plot(waste_range, waste_low,    label='Low',    color='#2ecc71')
ax.plot(waste_range, waste_medium, label='Medium', color='#f39c12')
ax.plot(waste_range, waste_high,   label='High',   color='#e74c3c')
ax.set_title('Output: Waste Level Score (0-100)')
ax.set_xlabel('Score'); ax.set_ylabel('Membership'); ax.legend(); ax.grid(alpha=0.3)

# Occupancy
ax = axes[1, 1]
ax.plot(occupancy_range, occ_empty,    label='Empty',    color='#e74c3c')
ax.plot(occupancy_range, occ_occupied, label='Occupied', color='#2ecc71')
ax.set_title('Input: Occupancy')
ax.set_xlabel('Level'); ax.set_ylabel('Membership'); ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('koswatt_membership_functions.png', dpi=150, bbox_inches='tight')
plt.show()
print("Plot saved to koswatt_membership_functions.png")